In [2]:
import os 
import json 
import networkx as nx 
import matplotlib.pyplot as plt 


from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader
)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.documents import Document


In [4]:
from dotenv import find_dotenv,load_dotenv

load_dotenv(find_dotenv(),override=True)

if os.environ["GROQ_API_KEY"]:
    print("Api key found ")
else:
    print('Api key not found ')

Api key found 


# **Defined Function To load Data**

In [7]:
file_path='impactOfCovid19.pdf'

def load_document(path):

    # Check the file the type and start that loader only 
    
    if path.endswith(".pdf"):
       loader=PyPDFLoader(path)
    elif path.endswith(".txt"):
        loader=TextLoader(path)
    elif path.endswith(".docx"):
        loader=Docx2txtLoader(path)
    else:
        raise ValueError("Unsupported Format")
    
    # Load the documents now
    documents=loader.load()
    return documents



In [8]:
documents=load_document(file_path)
print(f"Loaded {len(documents)} document pages")

Loaded 26 document pages


In [9]:
documents[0].page_content

'NBER WORKING PAPER SERIES\nTHE IMPACT OF COVID-19 ON STUDENT EXPERIENCES AND EXPECTATIONS:  \nEVIDENCE FROM A SURVEY\nEsteban M. Aucejo\nJacob F. French\nMaria Paola Ugalde Araya\nBasit Zafar\nWorking Paper 27392\nhttp://www.nber.org/papers/w27392\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nJune 2020\nNoah Deitrick and Adam Streff provided excellent research assistance. All errors that remain are \nours. The views expressed herein are those of the authors and do not necessarily reflect the views \nof the National Bureau of Economic Research.\nNBER working papers are circulated for discussion and comment purposes. They have not been \npeer-reviewed or been subject to the review by the NBER Board of Directors that accompanies \nofficial NBER publications.\n© 2020 by Esteban M. Aucejo, Jacob F. French, Maria Paola Ugalde Araya, and Basit Zafar. All \nrights reserved. Short sections of text, not to exceed two paragraphs, may be quoted without \ne

# **Chunk the Document**

In [10]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks=splitter.split_documents(documents)

print(f'Total Chunks:{len(chunks)}')

Total Chunks:159


In [11]:
chunks[0].page_content

'NBER WORKING PAPER SERIES\nTHE IMPACT OF COVID-19 ON STUDENT EXPERIENCES AND EXPECTATIONS:  \nEVIDENCE FROM A SURVEY\nEsteban M. Aucejo\nJacob F. French\nMaria Paola Ugalde Araya\nBasit Zafar\nWorking Paper 27392\nhttp://www.nber.org/papers/w27392\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nJune 2020\nNoah Deitrick and Adam Streff provided excellent research assistance. All errors that remain are'

# **Embeddings**

In [13]:
embedding_model=HuggingFaceBgeEmbeddings(
    model_name="all-miniLM-L6-v2"
)
print("Embedding Model Loaded ")

Embedding Model Loaded 


In [15]:
vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory='./chroma_db'
)

retriever=vector_store.as_retriever(search_kwargs={"k":3})

print('Chroma_DB Created Successfully')

Chroma_DB Created Successfully
